In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

import os

# load data

In [ ]:
# Initialize an empty list to hold the patches
neg_patches_list = []

# Loop through each of the file parts
for i in range(1, 5):  # Loops from 1 to 1
    filename = f'./drive/MyDrive/@antigen_epitope/abdb_v4/fold_5/data_processed_step1_with_SI/cnn_negative_patch_3points_part{i}.npy'
    patches_part = np.load(filename)
    neg_patches_list.append(patches_part)

# Concatenate the list of patches arrays into one array
neg_patches = np.concatenate(neg_patches_list, axis=0)
nan_mask = np.any(np.isnan(neg_patches), axis=(1, 2))
neg_patches = neg_patches[~nan_mask]
print(neg_patches[0:2])

In [ ]:
pos_patches = np.load('./drive/MyDrive/@antigen_epitope/abdb_v4/fold_5/data_processed_step1_with_SI/cnn_positive_patch_3points.npy')

nan_mask = np.any(np.isnan(pos_patches), axis=(1, 2))
pos_patches = pos_patches[~nan_mask]
print(pos_patches[0:2])

In [ ]:
np.set_printoptions(linewidth=np.inf, threshold=np.inf)
neg_patches[0]

In [ ]:
print(len(pos_patches), len(neg_patches))

In [ ]:
patches = arr = np.concatenate((pos_patches, neg_patches), axis=0)

labels = []
for patch in pos_patches:
    labels.append(1)
for patch in neg_patches:
    labels.append(0)

labels = np.array(labels)

In [ ]:
patches.shape, labels.shape

In [ ]:
type(labels),type(patches)

In [ ]:
data_func = np.ravel(labels).reshape(len(labels))  ## convert dataframe to array

data_seq = np.array(patches).reshape(len(patches),1,15,20)  ## reshape to 4d!

In [ ]:
data_func.shape, data_seq.shape

# Train Test split

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import TensorDataset, Dataset, DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
#Hyper Parameters
BATCH_SIZE = 500
LR = 0.001

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(data_seq, data_func,
                                                    test_size=0.2,
                                                    stratify = data_func,
                                                    random_state=1000)

X_train = torch.Tensor(X_train) # transform to torch tensor
X_test  = torch.Tensor(X_test) # transform to torch tensor
y_train = torch.Tensor(y_train) # transform to torch tensor
y_test  = torch.Tensor(y_test)

y_train = y_train.type(torch.FloatTensor)
y_test = y_test.type(torch.FloatTensor)

train_dataset = TensorDataset(X_train,y_train) # create your datset
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE, shuffle=True) # create your dataloader

test_dataset = TensorDataset(X_test,y_test) # create your datset
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True) # create your dataloader

In [ ]:
for i in train_loader:
    print(i[0][0][0])

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Sequential(    # input shape (1,15,20)
            nn.Conv2d(
                in_channels=1,         # input height: layers
                out_channels=16,       # n fileters
                kernel_size=(3,3),         # filter size
                stride=1,              # filter movement/step
                padding=(1,1),             # want same width and length of this image after Conv2d
            ),                         # ==> (16, 15, 20)
            nn.BatchNorm2d(16),        #applying batch norm
            nn.ReLU(),                 # activation
            nn.MaxPool2d(kernel_size=(3,2)), # choose max value in 2x2 area, output shape
        )                              # output shape (16,5,10)
        self.conv2 = nn.Sequential(    # input shape (16,5,10)
            nn.Conv2d(16,32,(2,2),1,(1,1)),    # output shape(32,5,10)
            nn.BatchNorm2d(32),        #applying batch norm
            nn.ReLU(),                 # activation
            nn.MaxPool2d(2),           # output shape (32,3,5)
        )
        self.fc1 = nn.Linear(32 * 3 * 5, 128)
        self.fc2 = nn.Linear(128, 32)
        self.out = nn.Linear(32, 1) # fully connected layer, output 10 classes

    def forward(self,x):
        x = self.conv1(x)
        x = self.conv2(x)              # (batch, 32, 7, 7)
        x = x.view(x.size(0), -1)      # flatten the output of conv2 to (batch_size, 32*7*7)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        output = torch.sigmoid(self.out(x))
        return output

In [ ]:
model_cnn = CNN().to(device)
print(model_cnn)

In [ ]:
# get the number of parameters in the model

def get_n_params(model):
    pp=0
    for p in list(model.parameters()):
        nn=1
        for s in list(p.size()):
            nn = nn*s
        pp += nn
    return pp

get_n_params(model_cnn)


from torchsummary import summary
summary(model_cnn,(1,15,20))

In [ ]:
LR = 0.0001
optimizer = torch.optim.Adam(model_cnn.parameters(), lr=LR)
loss_func = nn.BCELoss()

In [ ]:
train_losses=[]
train_accu=[]

# Set NumPy print options to display the full array
np.set_printoptions(threshold=np.inf)

def train(epoch):
    print('\nEpoch: %d'%epoch)

#    cnn.train()

    running_loss=0
    correct=0
    total=0

    for data in tqdm(train_loader):

        inputs, labels = data[0].to(device), data[1].to(device)
        np.set_printoptions(threshold=np.inf)
        #print(inputs)
        #print(labels)

        optimizer.zero_grad()
        outputs= torch.squeeze(model_cnn(inputs))  # touch.squeeze to match the dimension with labels
        loss=loss_func(outputs,labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        predicted = outputs.round()        # (outputs>0.5).float()
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss=running_loss/len(train_loader)
    accu=100.*correct/total

    train_accu.append(accu)
    train_losses.append(train_loss)
    print('Train Loss: %.3f | Accuracy: %.3f'%(train_loss,accu))
    output_accu = accu
    return float(output_accu)

In [ ]:
eval_losses=[]
eval_accu=[]

def test(epoch):
#    cnn.eval()

    running_loss=0
    correct=0
    total=0

    with torch.no_grad():
        for data in tqdm(test_loader):
            images,labels=data[0].to(device),data[1].to(device)

            outputs= torch.squeeze(model_cnn(images))

            loss= loss_func(outputs,labels)
            running_loss+=loss.item()

            predicted = outputs.round()    # output 0. or 1.
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    test_loss=running_loss/len(test_loader)
    accu=100.*correct/total

    eval_losses.append(test_loss)
    eval_accu.append(accu)

    print('Test Loss: %.3f | Accuracy: %.3f'%(test_loss,accu))
    output_accu = accu
    return float(output_accu)

In [ ]:
# reset the weights
def reset_weights(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        m.reset_parameters()

model_cnn.apply(reset_weights)

In [ ]:
class EarlyStopping():
    def __init__(self, tolerance=5, min_delta=0):
        self.tolerance = tolerance
        self.min_delta = min_delta
        self.counter = 0
        self.best_accu = None
        self.early_stop = False

    def __call__(self, val_accu):
        if self.best_accu is None:
            self.best_accu = val_accu
        elif val_accu < self.best_accu + self.min_delta:
            self.counter += 1
            if self.counter >= self.tolerance:
                self.early_stop = True
        else:
            self.best_accu = val_accu
            self.counter = 0


In [ ]:

early_stopping = EarlyStopping(tolerance=30, min_delta=5)

epoch = 100
for epoch in range(1, epoch+1):
    train_out_accu = train(epoch)
    test_out_accu = test(epoch)

    early_stopping(test_out_accu)
    if early_stopping.early_stop:
        print("Early stopping at epoch:", epoch)
        break

print('Finished Training')



In [ ]:
plt.plot(train_accu,'-o')
plt.plot(eval_accu,'-o')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.legend(['Train','Valid'])
plt.title('Train vs Valid Accuracy')

plt.show()

# Model Metrics

In [ ]:
def get_all_preds(model, loader):
    predictions = []   # output probability
    all_preds = []     # output 0 or 1
    all_true = []

    for batch in loader:
        images, labels = batch
        images, labels = images.to(device), labels.to(device)  # send the data to GPU

        outputs= torch.squeeze(model_cnn(images))
        predicted = outputs.round()  # output 0. or 1.

        predictions.append(outputs.cpu().numpy())  # output probability

        all_preds.append(predicted.cpu().numpy())
        all_true.append(labels.cpu().numpy())

    return all_preds, all_true, predictions

In [ ]:
with torch.no_grad():
    prediction_loader = torch.utils.data.DataLoader(test_dataset, batch_size=10000)
    y_pred,y_true,predictions = get_all_preds(model_cnn, prediction_loader)

predictions = np.concatenate(predictions).ravel()  # flatten out

y_pred = np.concatenate(y_pred).ravel()  # flatten out
y_true = np.concatenate(y_true).ravel()  # flatten out

In [ ]:
# classification report
report_string = classification_report(y_true,y_pred)
#report_string = classification_report(y_test,y_pred, target_names=category_names)

print(report_string)

In [ ]:
# Confusion matrix
confmat = confusion_matrix(y_true, y_pred)
confmat

In [ ]:
import seaborn as sns

def plot_cm(y_true, y_pred, figsize=(6,5)):
    cm = confusion_matrix(y_true, y_pred, labels=np.unique(y_true))
    cm_sum = np.sum(cm, axis=1, keepdims=True)
    cm_perc = cm / cm_sum.astype(float) * 100
    annot = np.empty_like(cm).astype(str)
    nrows, ncols = cm.shape
    for i in range(nrows):
        for j in range(ncols):
            c = cm[i, j]
            p = cm_perc[i, j]
            if i == j:
                s = cm_sum[i]
                annot[i, j] = '%.1f%%\n%d/%d' % (p, c, s)
            elif c == 0:
                annot[i, j] = ''
            elif i<j:
                annot[i, j] = '%.1f%%\n%d\nFP' % (p, c)
            else:
                annot[i, j] = '%.1f%%\n%d\nFN' % (p, c)
    cm = pd.DataFrame(cm, index=np.unique(y_true), columns=np.unique(y_true))
    ###cm = pd.DataFrame(cm, index=category_names, columns=category_names)
    cm.index.name = 'Actual'
    cm.columns.name = 'Predicted'
    fig, ax = plt.subplots(figsize=figsize)
    sns.set(font_scale=1.4)
    sns.heatmap(cm, cmap= "YlGnBu", annot=annot, fmt='', ax=ax, annot_kws={"fontsize":12})

In [ ]:
plot_cm(y_true, y_pred)

In [ ]:
from sklearn.metrics import roc_curve, auc, roc_auc_score

# calculate scores
auc = roc_auc_score(y_true, predictions)
# summarize scores
print('ROC AUC=%.3f' % (auc))
# calculate roc curves

In [ ]:
ns_fpr, ns_tpr, _ = roc_curve(y_true, predictions)

# plot the roc curve for the model
plt.plot(ns_fpr, ns_tpr, linestyle='--', label="AUC = "+ str(auc))
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
# axis labels
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
# show the legend
plt.legend()
# show the plot
plt.show()

In [ ]:
save_dir = './drive/MyDrive/@antigen_epitope/abdb_v4/fold_5/data_processed_step1_with_SI/'
torch.save(model_cnn, os.path.join(save_dir, 'model_cnn.pth'))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
ns_fpr, ns_tpr, thresholds = roc_curve(y_true, predictions)
# Calculate Youden's J statistic for each threshold
J = ns_tpr - ns_fpr

# Get the index of the maximum J value
idx = np.argmax(J)

# Get the best threshold
best_threshold = thresholds[idx]

# Calculate metrics
accuracy = accuracy_score(y_true, (predictions >= best_threshold).astype(int))
precision = precision_score(y_true, (predictions >= best_threshold).astype(int))
recall = recall_score(y_true, (predictions >= best_threshold).astype(int))
f1 = f1_score(y_true, (predictions >= best_threshold).astype(int))
auc = roc_auc_score(y_true, predictions)

# Display metrics
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Best Cutoff (Threshold): {best_threshold:.3f}")